# Домашнее задание. RAG — Туристический помощник

**Что внутри:** EDA → очистка выбросов → гибридный поиск (dense + BM25 + RRF) → реранкинг (ragatouille ColBERT и cross-encoder) → RAG на Qwen2.5 → PCA/UMAP → собственная реализация RAGAS-метрик → ablation.

Вся логика лежит в пакете `touristrag` (см. `tourist-rag/src`), ноутбук — тонкий «отчёт», который его вызывает.
Рекомендуется Colab с GPU (T4). Все ячейки с метриками нужно **выполнить**, чтобы в ноутбуке остались результаты.

## Установка и импорт библиотек

In [ ]:
# Colab: загрузите tourist-rag.zip в файлы сессии (или сделайте git clone своего репозитория)
!unzip -oq tourist-rag.zip
!pip install -q -e ./tourist-rag
!pip install -q sentence-transformers chromadb umap-learn plotly bitsandbytes accelerate gdown pyarrow
# ColBERT-реранкер из условия задания (необязательно). ragatouille 0.0.9 в свежем Colab падает на `langchain.retrievers`,
# поэтому ставим langchain-classic и подключаем шим (ячейка ниже). Если не взлетит — ноутбук пойдёт дальше без ColBERT.
!pip install -q ragatouille langchain-classic

In [ ]:
import os, sys, inspect
for p in ("tourist-rag/src", "../src"):
    if os.path.isdir(p): sys.path.insert(0, os.path.abspath(p))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from touristrag.config import Config
from touristrag.utils import set_seed

# шим для ragatouille: устаревшие модули langchain переехали в langchain_classic (best-effort)
try:
    import importlib
    for m in ("retrievers", "retrievers.document_compressors", "retrievers.document_compressors.base"):
        sys.modules.setdefault(f"langchain.{m}", importlib.import_module(f"langchain_classic.{m}"))
except Exception as e:
    print("шим langchain пропущен:", repr(e)[:120])

pd.set_option("display.max_colwidth", 120)
cfg = Config()          # все гиперпараметры в одном месте
set_seed(cfg.seed)
cfg

## Загрузка данных (4 балла)

In [ ]:
from touristrag.data import load_raw, detect_schema, standardize, decode_image

raw = load_raw(cfg.data_path, cfg.data_url)
print(raw.shape)
raw.head(3)

Названия колонок определяем автоматически; если что-то определилось неверно — задайте `cfg.column_overrides`.

In [ ]:
schema = detect_schema(raw.columns, cfg.column_overrides)
print(schema)
assert schema["caption"], "BLIP-подписи не найдены: задайте cfg.column_overrides = {'caption': '<имя колонки>'}"
df = standardize(raw, schema)
df.drop(columns="image").head()

### EDA

In [ ]:
from touristrag.eda import overview, plot_eda, show_images

from touristrag.data import place_key
df["place_key"] = place_key(df)
display(overview(df))
plot_eda(df); plt.show()

In [ ]:
show_images(df, n=6); plt.show()

**Одна достопримечательность — много записей.** Склеиваем их в «места» (по WikiData id, иначе по название+город). Для описания есть две стратегии из условия:
`longest` (одно самое длинное) и `concat` (до 3 разных описаний, near-duplicates отбрасываются по Жаккару).

In [ ]:
from touristrag.data import build_places

places_all = build_places(df, cfg.desc_strategy, cfg.max_descriptions, cfg.max_captions)
places_longest = build_places(df, "longest")
print(f"{len(df)} записей -> {len(places_all)} уникальных мест")
cmp = pd.DataFrame({"len_longest": places_longest.description.str.len(), "len_concat": places_all.description.str.len()})
print(cmp.describe().round(1))
places_all.drop(columns="image").head()

### Очистка выбросов (TF-IDF + правила + zero-shot по эмбеддингам)

In [ ]:
from touristrag.cleaning import tfidf_terms, flag_outliers, review_table, drop_places, stratified_sample

texts = (places_all.name + ". " + places_all.description + " " + places_all.caption).tolist()
terms = tfidf_terms(texts, top_n=30)
display(terms["frequent"].T); display(terms["rare"].T)

In [ ]:
from touristrag.retrieval import Embedder
embedder = Embedder(cfg.embed_model)

flagged = flag_outliers(places_all, embedder)
print("помечено как мусор:", int(flagged.is_junk.sum()), "из", len(flagged))
review_table(flagged, 40)

**Ручная проверка** (требуется в условии): смотрим картинки самых подозрительных мест и при необходимости дополняем `BLACKLIST`.

In [ ]:
def show_places(frame, n=12, cols=4):
    frame = frame[frame.image != ""].head(n)
    rows = (len(frame) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3.6*rows))
    for ax in np.atleast_1d(axes).ravel(): ax.axis("off")
    for ax, r in zip(np.atleast_1d(axes).ravel(), frame.itertuples()):
        ax.imshow(decode_image(r.image)); ax.set_title(f"{r.name[:26]}\n{r.caption[:38]}", fontsize=8)
    plt.tight_layout(); plt.show()

show_places(flagged.sort_values("junk_score", ascending=False), n=12)

In [ ]:
BLACKLIST = []   # place_id, которые вы нашли руками и хотите убрать дополнительно
places = drop_places(flagged[~flagged.is_junk].drop(columns=["rule_flag","tfidf_atyp","zs_margin","junk_score","is_junk"]), BLACKLIST)
print("чистых мест:", len(places))

**Чистая тестовая выборка:** ≥100 разных мест, равномерно по городам (round-robin по страте «город», по одному месту на объект).

In [ ]:
test_places = stratified_sample(places, n=100, by="city", seed=cfg.seed)
print(len(test_places), "мест,", test_places.city.nunique(), "городов")
test_places.city.value_counts().head(10)

## RAG

**Решения, отличающиеся от семинара**
* эмбеддинги `multilingual-e5-base` (русский + английские BLIP-подписи) вместо английской gte-small;
* векторная БД **Chroma** вместо FAISS, плюс **BM25** (с «псевдо-стеммером» для русской морфологии) и слияние **RRF**;
* чанки по предложениям с **контекстным заголовком** «Название (Город).»; подпись BLIP — отдельный «визуальный» чанк;
* реранкинг: **ragatouille (ColBERT)** и multilingual cross-encoder `bge-reranker-v2-m3`; выбор — по метрикам (ablation ниже);
* порядок контекста «лучшие по краям» (lost-in-the-middle).

In [ ]:
from touristrag.pipeline import build_index, TouristRAG
from touristrag.retrieval import HybridRetriever, Reranker

os.makedirs("artifacts", exist_ok=True)
chunks, store, chunk_emb = build_index(places, embedder, cfg, store_path="artifacts/chroma")
ch = pd.DataFrame(chunks)
print(len(places), "мест ->", len(ch), "чанков;", ch.chunk_type.value_counts().to_dict())
ch.text.map(embedder.count_tokens).describe().round(1)

In [ ]:
retriever = HybridRetriever(embedder, store, chunks, cfg.rrf_k)
rr_ce = Reranker("cross-encoder", cfg.rerank_model)
try:
    rr_colbert = Reranker("colbert")           # ragatouille, как в условии
except Exception as e:
    rr_colbert = None; print("ragatouille недоступен:", repr(e)[:200])

from touristrag.llm import HFChat
llm = HFChat(cfg.llm_model, cfg.llm_4bit, cfg.max_new_tokens)
rag = TouristRAG(retriever, rr_ce, llm, cfg)

In [ ]:
q = "Какой кремль находится в Нижнем Новгороде?"
for name, rr in [("без реранка", Reranker("none")), ("cross-encoder", rr_ce), ("ColBERT (ragatouille)", rr_colbert)]:
    if rr is None: continue
    hits = rr.rerank(q, retriever.retrieve(q, k=cfg.k_retrieve), top_k=5)
    print(f"{name:24s}", [h["name"] for h in hits])
res = rag.answer(q); print("\n", res["answer"])

### Визуализация эмбеддингов: PCA и UMAP (обучаются независимо)

In [ ]:
from touristrag.viz import reduce_embeddings, plot_embeddings

mask = (ch.chunk_type == "description").to_numpy()
E, meta = chunk_emb[mask], ch[mask].reset_index(drop=True)
pca_xy, pca = reduce_embeddings(E, "pca")
umap_xy, umap_model = reduce_embeddings(E, "umap")
ev = pca.explained_variance_ratio_.sum()
f1 = plot_embeddings(pca_xy, meta.city, meta.name, f"PCA (объясн. дисперсия {ev:.1%})")
f2 = plot_embeddings(umap_xy, meta.city, meta.name, "UMAP (cosine)")
os.makedirs("results", exist_ok=True); f1.write_html("results/pca.html"); f2.write_html("results/umap.html")
f1.show(); f2.show()

# RAGAS (3 балла)

> Датасет охватывает 4 города (Екатеринбург, Ярославль, Владимир, Нижний Новгород), поэтому тестовые вопросы — только про них.

Метрика **answer relevancy** реализована вручную (`touristrag.evaluation.answer_relevancy`):
LLM по ответу восстанавливает `N=3` вопроса → косинус эмбеддингов с исходным вопросом → среднее; уклончивый ответ (`noncommittal`) = 0.
«Уклончивость» определяется судьёй **и** правилом `is_refusal` (малые модели пропускают вежливые отказы). Если судья вернул невалидный JSON и вопросы не восстановить из текста, результат `NaN`, а не 0 — сбой судьи не должен выглядеть как плохой ответ.

Датасет в формате RAGAS собирает `data_preprocess`: вопросы генерирует LLM по описанию (половина — «слепые», без названия места).

In [ ]:
from touristrag.evaluation import (data_preprocess, run_rag_on_dataset, answer_relevancy, summarize,
                                   faithfulness, context_precision, context_recall, evaluate_retrieval)
print(inspect.getsource(data_preprocess)[:1800]); print(inspect.getsource(answer_relevancy))

In [ ]:
llm_fn   = lambda p: llm(p)
embed_fn = lambda xs: embedder.encode(xs, kind="query")

ds = data_preprocess(places, llm_fn, n=cfg.n_eval, blind_ratio=cfg.blind_ratio, seed=cfg.seed)
ds[["place_id","qtype","question","ground_truth"]].head(8)

In [ ]:
ds = run_rag_on_dataset(ds, rag)
ar, details = answer_relevancy(ds.question, ds.answer, llm_fn, embed_fn, n_gen=3, return_details=True)
ds["answer_relevancy"] = ar
print(f"answer_relevancy = {np.nanmean(ar):.3f}   (валидных {np.isfinite(ar).sum()}/{len(ar)})")
print(ds.groupby("qtype").answer_relevancy.mean().round(3).to_dict())
ds[["question","answer","answer_relevancy"]].sort_values("answer_relevancy").head(5)

Вопросы на карте эмбеддингов: насколько тестовые запросы лежат рядом со «своими» местами.

In [ ]:
q_emb = embedder.encode(ds.question.tolist(), kind="query")
f = plot_embeddings(umap_xy, meta.city, meta.name, "UMAP: места и тестовые вопросы (×)",
                    query_coords=umap_model.transform(q_emb), query_hover=ds.question)
f.write_html("results/umap_queries.html"); f.show()

# Протестируйте ваш RAG (3 балла)

**Ablation retrieval-части** — Hit@k / MRR по эталонному месту; отдельно для вопросов с названием (`named`) и «слепых» (`blind`).

In [ ]:
def run_ablation(ds):
    variants = {"dense": ("dense", Reranker("none")), "bm25": ("bm25", Reranker("none")),
                "hybrid (RRF)": ("hybrid", Reranker("none")), "hybrid + cross-encoder": ("hybrid", rr_ce)}
    if rr_colbert is not None: variants["hybrid + ColBERT (ragatouille)"] = ("hybrid", rr_colbert)
    rows = {}
    for name, (mode, rr) in variants.items():
        rag.reranker = rr
        res = evaluate_retrieval(lambda q: rag.rank_places(q, mode=mode, rerank=rr.backend != "none"), ds)
        rows[name] = {**res["all"], **{f"mrr[{t}]": res[t]["mrr"] for t in res if t != "all"}}
    rag.reranker = rr_ce
    return pd.DataFrame(rows).T.round(3)

ablation = run_ablation(ds); ablation.to_csv("results/ablation.csv"); ablation

In [ ]:
ablation[["hit@1","hit@3","hit@5","mrr"]].plot.bar(figsize=(10,4), rot=20, title="Retrieval ablation"); plt.tight_layout(); plt.show()

**Разбор ошибок:** где эталонное место не попало в топ-3 и что нашлось вместо него.

In [ ]:
rag.reranker = rr_ce
names = places.set_index("place_id").name
bad = ds[[r.place_id not in r.retrieved_place_ids[:3] for r in ds.itertuples()]]
print(f"промахов top-3: {len(bad)}/{len(ds)}")
for r in bad.head(5).itertuples():
    print(f"\n[{r.qtype}] {r.question}\n  gold: {r.name}\n  got : {[names[i] for i in r.retrieved_place_ids[:3]]}")

In [ ]:
for q in ["Что такое Золотые ворота во Владимире?", "Какая башня в Екатеринбурге белого цвета?", "Расскажи про соборную мечеть в Нижнем Новгороде"]:
    r = rag.answer(q); print("Q:", q, "\nA:", r["answer"], "\n   источники:", [h["name"] for h in r["hits"]], "\n")

# Дополнительная часть (+3 балла)

Реализованы все retrieval-метрики RAGAS на основе LLM-судьи (`touristrag.evaluation`): **context precision** (AP@K, награждает релевантные чанки в начале), **context recall**, а также **faithfulness**.
Плюс метрики без LLM: Hit@k и MRR (таблица выше). Считаем на подвыборке из 30 вопросов (судья дорогой).

In [ ]:
sub = ds.sample(30, random_state=cfg.seed)
scores = {
    "answer_relevancy":  ds.answer_relevancy.to_numpy(),
    "faithfulness":      faithfulness(sub.question, sub.answer, sub.contexts, llm_fn),
    "context_precision": context_precision(sub.question, sub.ground_truth, sub.contexts, llm_fn),
    "context_recall":    context_recall(sub.ground_truth, sub.contexts, llm_fn),
}
summary = summarize(scores); summary.round(3)

In [ ]:
ds.to_csv("results/eval_samples.csv", index=False); summary.to_csv("results/ragas_summary.csv")
import json; json.dump({"config": cfg.__dict__, "ablation": ablation.to_dict("index"), "ragas": summary["mean"].to_dict()},
                       open("results/metrics.json","w"), ensure_ascii=False, indent=2, default=str)

## Выводы
*(заполните после запуска: какой вариант выиграл в ablation и почему, насколько `blind`-вопросы сложнее `named`, типичные ошибки из разбора, ограничения.)*

**Ограничения:** вопросы и эталоны сгенерированы по тем же описаниям, что лежат в индексе, поэтому метрики оптимистичны; судья — небольшая LLM, возможен шум (поэтому `NaN` при невалидном JSON и подвыборка для дорогих метрик).